In [ ]:
"""
HEALTHCARE READMISSION ANALYTICS PROJECT
-------------------------------------------
Author : Khushi 
Date : May 2026
Purpose : End to end analysis of hospital readmissions using diabetes patient data

This script demonstrates:
1. Data acquisition from public sources 
2. Data preprocessing and cleaning
3. Exploratory data analysis
4. Predictive Modelling 
5. Results visualization and reporting

Key expectations from this project:
1. Data acquisition from public sources
2. Data preprocessing and cleaning
3. Exploratory data analysis
4. Predictive modeling
5. Results visualization and reporting
"""

# ========================================================================
# SECTION 1: ENVIRONMENT SETUP AND LIBRARY IMPORTS
# ========================================================================

# DATA MANIPULATION AND ANALYSIS
import pandas as pd
import numpy as np

# VISUALISATION LIBRARIES 
import matplotlib.pyplot as plt
import seaborn as sns

# MACHINE LEARNING LIBRARIES 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Utility Libraries 
import warnings
import os
from datetime import datetime

# Configuration Settings
warnings.filterwarnings('ignore') # Suppress warnings for cleaner output
pd.set_option('display.max_columns',None) # Display all columns
pd.set_option('display.max_rows',100) # Display upto 100 rows
plt.style.use('seaborn-v0_8-darkgrid') # Set consistent plot style

# Set random seed for reproducibility
np.random.seed(42)

print("="*80)
print ("HEALTHCARE READMISSION ANALYTICS PROJECT")
print("="*80)
print(f"Execution started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S ')}")
print ("All libraries imported succesfully! \n")




In [ ]:
# ========================================================
# DATA ACQUISITION
# ========================================================

"""
Download the Diabetes 130-US hospitals dataset from UCI ML Repository.
This dataset contains 10 years of hospital admission records for diabetes patients.

Dataset characteristics:
- 101,766 hospital admissions
- ~50 features (Patient demographics, diagnoses, mediactions, procedures)
- Target variable: Readmission status
"""

print ("\n" + "=" * 80)
print("Data Acquisition")
print("=" * 80)

# Define data URL 
Data_url = "https://archive.ics.uci.edu/dataset/34/diabetes" 

"""
DATA DOWNLOAD INSTRUCTIONS:
--------------------------------------------------------
1. Visit UCI Machine Learning Website for diabetes datasets 
2. Click 'Download' to get 'database_diabetes.zip'
3. Extract the zip file
4. Place 'diabetic_data.csv' file in your working directory
"""

# Check if data exists
data_file = r"C:\Users\hp1\Downloads\diabetic_data.csv"

if os.path.exists(data_file):
    print(f"Data File '{data_file}' found in current directory.")

else:
    print(f"Data file '{data_file}' not found.")
    print("Please download the file as per the instructions given above")

# Load the dataset
try:
    print("\n Loading dataset....")
    df_raw = pd.read_csv(data_file)
    print("Dataset loaded succesfully")
    print(f" - Shape: {df_raw.shape[0]:,} rows * {df_raw.shape[1]} columns")
    print(f" - Memory usage: {df_raw.memory_usage(deep = True).sum() / 1024**2:.2f} MB")

except FileNotFoundError:
    print("Check for the file if it is downloaded correctly or not")

# Display basic information about dataset
print ("\n" + "-" * 80)
print("Initial Data Preview")
print("-" * 80)
print("\n First 5 rows:")
print(df_raw.head())

print("\n Dataset Info:")
print(df_raw.info())

print("\n Basic Statistics:")
print(df_raw.describe())

In [ ]:
# ===================================================================
# SECTION 3: DATA QUALITY ASSESMENT
# ===================================================================

"""
Before any analysis, identify data quality issues:
1. Missing Values
2. Duplicate Records
3. Data Types
4. Value Distributions
5. Outliers

"""
# Create output directory for visualisation
import os 
if not os.path.exists('visualisations'):
    os.makedirs('visualisations')
    print("Created Visualisations directory for output")

print("\n\n" + "=" * 80)
print("DATA QUALITY ASSESMENT")
print("=" * 80)

# Create a copy for preprocessing
df = df_raw.copy()

# 3.1 - Check for duplicate records
print("\n 1. DUPLICATE RECORDS CHECK")
print("-" * 80)
duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates:,}")

if duplicates > 0:
    print(f"Percentage of duplicates: {duplicates/len(df)*100:.2f}%")

# 3.2 - Missing Value Analysis
print ("\n\n 2. MISSING VALUES ANALYSIS")
print("-" * 80)

# Calculate Missing Values
missing_data = pd.DataFrame({
    "Column":df.columns,
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df) * 100). round(2) 
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values(
    'Missing_Percentage', ascending = False
)

if len(missing_data) > 0 :
    print(f"\n Columns with missing values: {len(missing_data)}")
    print("\n" + missing_data.to_string(index = False))

else:
    print("No missing values detected in dataset !")

# 3.3 - Check for placeholder values
print("\n \n 3. PLACEHOLDER VALUES CHECK")
print("-" * 80)
print("Healthcare datasets often use '?' or 'Unknown' as placeholders.")

# Count '?' occurences in each column
placeholder_counts = {}
for col in df.columns :
    if df[col].dtype == 'object': # checking only string columns
        placeholder_count = (df[col] == '?').sum()
        if placeholder_count > 0:
            placeholder_counts[col] = {
                'count' : placeholder_count,
                'percentage': round(placeholder_count/len(df) * 100,2)
            }

if placeholder_counts :
    print(f" \n Columns with '?' placeholders: {len(placeholder_counts)}")
    for col, stats in sorted (placeholder_counts.items(),
                             key = lambda x: x[1]['percentage'],
                             reverse = True):
        print (f" - {col}: {stats['count']:,} ({stats['percentage']}%)")

else:
    print("No '?' placeholders found")

# Data Types examination
print("\n \n 4. DATA TYPES SUMMARY")
print("-" * 80)
dtype_summary = df.dtypes.value_counts()
print(dtype_summary)

print ("\n \n Columns by data types:")
for dtype in df.dtypes.unique ():
    cols = df.select_dtypes(include = [dtype]).columns.tolist()
    print(f"\n {dtype}:")
    print(f"Count: {len(cols)}")
    print(f"Columns: {','.join(cols[:5])}{'...' if len (cols) > 5 else ''}")

# 3.5 - Target Variable Distribution 
print("\n \n 5. TARGET VARIABLE ANALYSIS")
print("-" * 80)
print("Our target variable is 'readmitted' - indicating if patient was readmitted. ")

if "readmitted" in df.columns:
    print("\n Readmission Status distribution:")
    readmit_dist = df['readmitted'].value_counts()
    print(readmit_dist)
    print("\n Percentages:")
    print((readmit_dist / len(df) * 100).round (2))

# Visualize Target Distribution
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
readmit_dist.plot(kind='bar', color=['#2ecc71','#e74c3c','#3498db'])
plt.title('Readmission Status Distribution', fontsize = 14, fontweight='bold')
plt.xlabel('Readmission Status')
plt.ylabel('Count')
plt.xticks(rotation = 0)

plt.subplot(1,2,2)
plt.pie(readmit_dist.values, labels = readmit_dist.index, autopct = '%1.1f%%',
        colors = ['#2ecc71','#e74c3c','#3498db'])
plt.title ('Readmission Percentage', fontsize = 14, fontweight = 'bold')
plt.tight_layout()
plt.savefig('visualisations/01_target_distribution.png', dpi = 300, bbox_inches= 'tight')
print("\n Visualisation Saved : visualisations/01_target_distribution.png")
plt.show()



    

In [ ]:
# =============================================================
# SECTION 4: DATA CLEANING AND PREPROCESSING
# =============================================================
"""
Based on thr quality assesment, let's clean the data:
1. Handle missing/placeholder values
2. Remove unnecessary columns
3. Convert data types
4. Create binary target variable
"""

print("\n \n" + "=" * 80)
print("DATA CLEANING AND PREPROCESSING")
print("=" * 80)

# 4.1 Replace '?' with NaN for consistent missing value handling
print("\n 1. REPLACING PLACEHOLDER VALUES")
print ("-" * 80)
df = df.replace('?',np.nan)
print("Replaced all '?' with NaN")

# 4.2 Drop columns with excessive missing data or low predictive value
print("\n 2. REMOVING PROBLEMATIC COLUMNS")
print("-" * 80)

# Columns to drop
columns_to_drop = [
    'encounter_id' ,   # Identifier, not predictive
    'patient_nbr',   # Identifier, not predictive
    'weight',    # 97% missing
    'payer_code',   # 40% missing, many categories
    'medical_specialty',   # 50% missing
]

print(f"Dropping {len(columns_to_drop)} columns:")
for col in columns_to_drop:
    if col in df.columns:
        missing_pct = (df[col].isnull().sum() / len(df) * 100)
        print(f" - {col}: {missing_pct:.1f}% missing")
        df = df.drop(columns=[col])

print(f"\n New dataset shape: {df.shape[0]:,} rows * {df.shape[1]} columns")

# 4.3 Handle remaining missing values
print("\n 3. HANDLING REMAINING MISSING VALUES")
print("-" * 80)

# For race, gender, diagnosis codes: fill with 'Unknown'
categorical_fills_cols = ['race','gender','diag_1','diag_2','diag_3']

for col in categorical_fills_cols :
    if col in df.columns:
        missing_before = df[col].isnull().sum()
        df[col] = df[col].fillna("Unknown")
        print(f" - {col}: Filled {missing_before:,} missing values with Unknown")

print("Categorical missing values handled")

# Create binary target variable 
print("\n 4. CREATING BINARY TARGET VARIBALE")
print("-" * 80)
print ("Converting 3-class target to binary: Readmitted(<30 days) vs Not Readmitted(>30 days or No)")

if "readmitted" in df.columns:
    # <30 -> 1(High Risk) , >30 or 'NO' -> 0(Low Risk)
    df['readmitted_binary'] = df['readmitted'].apply(
        lambda x: 1 if x == '<30' else 0
    )

    print("\n Mapping:")
    print(" - '<30 days' -> (Readmitted - High Risk) ")
    print(" - 'NO' or '>30 days' -> 0 (Not Readmitted - Low Risk)")

    print("\n New Target distribution:")
    print(df['readmitted_binary'].value_counts())
    print("\n Percentages:")
    print((df['readmitted_binary'].value_counts() / len(df) * 100). round(2))

    # Calculate class imbalance ratio
    class_counts = df['readmitted_binary'].value_counts()
    imbalance_ratio = class_counts[0] / class_counts[1]
    print(f"\n Class imbalance ratio: {imbalance_ratio:.2f}:1")
    print("(This is important for model selection and evaluation)")

# 4.5 - Feature Engineering - Consolidate diagnosis codes 
print("/n 5. FEATURE ENGINEERING - DIAGNOSIS GROUPING")
print("-" * 80)
print("ICD - 9 diagnosis codes can be grouped into clinically meaningful categories")

def categorize_diagnosis(diag_code):
    """
    Groups ICD-9 diagnosis codes into clinical categories

    Paramters: 
    ------------------
    diag_code : str
        ICD-9 diagnosis code

    Returns:
    --------------
    str : Category Name
    """
    if pd.isna (diag_code) or diag_code == "Unknown":
       return 'Unknown'

    # Convert to string and extract numeric part
    diag_str = str(diag_code)

    # Try to extract numeric code 
    try:
        if diag_str.startswith('V') or diag_str.startswith('E'):
            return 'Other'

        code = float(diag_str)

        # ICD-9 code ranges
        if 390 <= code <= 459 or code == 785:
            return 'Circulatory'
        elif 460 <= code <= 519 or code == 786:
            return 'Respiratory'
        elif 520 <= code <= 579 or code == 787:
            return 'Digestive'
        elif 250 <= code <251:
            return 'Diabetes'
        elif 800 <= code <= 999:
            return 'Injury'
        elif 710 <= code <= 739:
            return 'Musculoskeletal'
        elif 580 <= code <= 629 or code == 788:
            return 'Genitourinary'
        elif 140 <= code <= 239:
            return 'Neoplasms'
        else :
            return 'Other'
    except :
        return 'Other'

# Apply diagnosis categorization
for diag_col in ['diag_1','diag_2','diag_3']:
    if diag_col in df.columns:
        new_col_name = f'{diag_col}_category'
        df[new_col_name] = df[diag_col].apply(categorize_diagnosis)
        print(f"Created {new_col_name}")
        print(f"Categories: {df[new_col_name].value_counts().to_dict()}")

print("\n Feature engineering completed")

# 4.6 - Convert age to numeric values 
print(f"\n 6. CONVERTING AGE TO NUMERIC")
print("-" * 80)

if 'age' in df.columns:
    print("Age columns consists ranges like '[70-80)'. Converting to mid-point values")

    # Define age range Mapping
    age_mapping = {
        '[0-10)' : 5,
        '[10-20)' : 15,
        '[20-30)' : 25,
        '[30-40)' : 35,
        '[40-50)' : 45,
        '[50-60)' : 55,
        '[60-70)' : 65,
        '[70-80)' : 75,
        '[80-90)' : 85,
        '[90-100)' : 95,
    }

    df['age_numeric'] = df['age'].map(age_mapping)
    print(f"Created 'age_numeric' column")
    print(f" Range: {df['age_numeric'].min()} - {df['age_numeric'].max()} years")

print("\n" + "=" * 80)
print("DATA CLEANING COMPLETE!")
print("=" * 80)
print(f"Final dataset shape : {df.shape[0]:,} rows * {df.shape[1]} columns")
print(f"Ready for exploratory analysis and modelling!")

# Save Cleaned dataset
output_file = 'diabetic_data_cleaned.csv'
df.to_csv(output_file, index = False)
print(f"\n Cleaned dataset saved to: '{output_file}'")







        

In [ ]:
# ==========================================================
# SECTION 5: CLEANED DATA SUMMARY
# ==========================================================

print("\n\n" + "=" * 80)
print("Cleaned Dataset Summary")
print("="*80)

# 5.1 Numeric Features Summary
print("\n 1. NUMERIC FEATURES SUMMARY")
print("-"*80)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Number of numeric features:{len(numeric_cols)}")
print("\n Key numeric features:")
key_numeric = ['time_in_hospital', 'num_lab_procedures','num_procedures',
               'num_medications','number_outpatient','number_emergency',
               'number_inpatient','number_diagnoses','age_numeric']

for col in key_numeric:
    if col in df.columns:
        print(f"\n{col}:")
        print(f"Mean: {df[col].mean():.2f}")
        print(f"Median: {df[col].median():.2f}")
        print(f"Std Dev: {df[col].std():.2f}")
        print(f"Range: [{df[col].min()}, {df[col].max()}]")

# 5.2 Categorical Features Summary
print("\n\n 2. CATEGORICAL FEATURES SUMMARY")
print("="*80)
categorical_col = df.select_dtypes(include=['object']).columns.tolist()
print(f"Number of categorical features: {len(categorical_col)}")

print("\nKey categorical features and their unique values:")
key_categorical = ['race', 'gender', 'admission_type_id', 'discharge_disposition_id',
                   'admission_source_id', 'diag_1_category', 'diabetesMed', 'change']

for col in key_categorical :
    if col in df.columns:
        unique_count = df[col].nunique()
        print(f"\n{col}:{unique_count} unique values")
        if unique_count <= 10:
            print(df[col].value_counts().head())

# Data Quality Final Check
print("\n \n 3. FINAL DATA QUALITY CHECK")
print("-"*80)
print(f"Total Records: {len(df):,}")
print(f"Total Features: {df.shape[1]}")
print(f"\n Missing Values Remaining:")
remaining_missing = df.isnull().sum().sum()
print(f"Total: {remaining_missing:,}")

if remaining_missing > 0:
    print(f" Percentage:{ remaining_missing/(df.shape[0] * df.shape[1])}")
else:
    print("No missing Values!")

print("Dataset is clean and ready for analysis")

In [ ]:
# ==============================================================
# SECTION 6 - EXPLORATORY DATA ANALYSIS(EDA)
"""
EDA helps us understand :
1. Indiviual features distributions
2. Relationship between features
3. Patterns related to readmission
4. Data-driven insights for healthcare stakeholders
"""
import matplotlib
matplotlib.rcParams['agg.path.chunksize'] = 1000
matplotlib.use('Agg') 

print("Unique age values:", df['age'].nunique())

print("\n \n" + "="* 80)
print("EXPLORATORY DATA ANALYSIS")
print("="* 80)
plt.close('all')

# 6.1 UNIVARIATE ANALYSIS - NUMERIC FEATURES
print("\n 1. UNIVARIATE ANALYSIS - NUMERIC FEATURES")
print("-"*80)

# Select key numeric features for analysis
numeric_features = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                   'num_medications', 'number_outpatient', 'number_emergency',
                   'number_inpatient', 'number_diagnoses', 'age_numeric']

# Create Distribution Plots
fig, axes = plt.subplots(3, 3, figsize=(18,14))
fig.suptitle('Distribution of Numeric Features', fontsize =16, fontweight='bold', y=1.00)

for idx, col in enumerate(numeric_features):
    if col in df.columns:
        row = idx // 3
        col_idx = idx % 3
        ax = axes[row, col_idx]

        # Histogram with KDE
        df[col].hist(bins=30, ax=ax, alpha=0.7, color='steelblue', edgecolor='black')
        ax.set_title(f'{col}\n(Mean: {df[col].mean():.2f}, Median: {df[col].median():.2f})',
                    fontsize =11)
        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')
        ax.grid(axis='y',alpha=0.3)

plt.tight_layout()
plt.savefig('visualisations/02_numeric_distributions.png', dpi= 300, bbox_inches = 'tight')
print("Saved: visualisations/02_numeric_distributions.png")
# plt.show()
plt.close('all')

# Statistical Summary
print("\n Statistical Summary of Numeric Features:")
print(df[numeric_features].describe().round(2))

# 6.2 UNIVARIATE ANALYSIS - CATEGORICAL FEATURES
print("\n\n 2. UNIVARIATE ANALYSIS - CATEGORICAL FEATURES")
print("-"* 80)

# Gender Distribution
fig, axes = plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Key Categoriacl Features Distribution', fontsize= 16, fontweight='bold')

# Gender 
if 'gender' in df.columns:
    gender_counts= df['gender'].value_counts()
    axes[0,0].bar(gender_counts.index, gender_counts.values, color=['#3498db','#e74c3c','#95a5a6'])
    axes[0,0].set_title('Gender Distribution', fontweight='bold')
    axes[0,0].set_ylabel('Count')
    # if len(gender_counts) <= 10:
    for i, v in enumerate(gender_counts.values):
        axes[0,0].text(i,v,f'{v:,}',ha='center',va='bottom')

# Race
if 'race' in df.columns:
    race_counts= df['race'].value_counts().head(6)
    axes[0,1].barh(race_counts.index, race_counts.values, color=['coral'])
    axes[0,1].set_title('Race Distribution (Top 6)', fontweight='bold')
    axes[0,1].set_xlabel('Count')
    for i, v in enumerate(race_counts.values):
        axes[0,1].text(v,i,f'{v:,}',va='center')

# Primary Diagnosis Category 
if 'diag_1_category' in df.columns:
    diag_counts= df['diag_1_category'].value_counts().head(8)
    axes[1,0].barh(diag_counts.index, diag_counts.values, color=['mediumseagreen'])
    axes[1,0].set_title('Primary Diagnosis Category (Top 8)', fontweight='bold')
    axes[1,0].set_xlabel('Count')
    for i, v in enumerate(diag_counts.values):
        axes[1,0].text(v,i, f'{v:,}',va='center')

# Age Distribution
if 'age' in df.columns:
    age_counts = df['age'].value_counts().sort_index().head(10)  # LIMIT
    axes[1, 1].bar(range(len(age_counts)), age_counts.values)
    axes[1, 1].set_title('Age Group Distribution', fontweight='bold')
    axes[1, 1].set_xticks(range(len(age_counts)))
    axes[1, 1].set_xticklabels(age_counts.index, rotation=45)
    axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('visualisations/03_categorical_distributions.png',dpi = 100)
print("Saved Visualisations: visualisations/03_categorical_distributions.png \n")
plt.close('all')



In [ ]:
# 6.3 BIVARIATE ANALYSIS - FEATURE VS TARGET
print("\n\n 3. BIVARIATE ANALYSIS - RELATIONSHIP WITH READMISSION")
print("-"*80)

# 6.3.1 Numeric Features vs Readmission
print("\n Analyzing numeric features by readmission status...")

fig,axes = plt.subplots(3,3,figsize=(18,14))
fig.suptitle('Numeric Features by Readmission Status', fontsize=16, fontweight='bold', y=1.00)

for idx, col in enumerate (numeric_features):
    if col in df.columns:
        row = idx//3
        col_idx = idx % 3
        ax = axes[row,col_idx]

        # Box plot compairirng readmitted vs not readmitted
        data_to_plot = [
            df[df['readmitted_binary'] == 0][col].dropna(),
            df[df['readmitted_binary'] == 1][col].dropna()
        ]

        bp = ax.boxplot(data_to_plot, labels=['Not Readmitted','Readmitted'],
                       patch_artist=True, showmeans = True)

        # Color the boxes 
        bp['boxes'][0].set_facecolor('lightgreen')
        bp['boxes'][1].set_facecolor('lightcoral')

        ax.set_title(f'{col}', fontsize= 11, fontweight='bold')
        ax.set_ylabel('Value')
        ax.grid(axis='y', alpha= 0.3)

        # Add mean values as text
        mean_0 = df[df['readmitted_binary'] == 0][col].mean()
        mean_1 = df[df['readmitted_binary'] == 1][col].mean()
        ax.text(1, mean_0, f'{mean_0:.1f}', ha = 'center', va='bottom',fontweight='bold')
        ax.text(2, mean_1, f'{mean_1:.1f}', ha = 'center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('visualisations/04_numeric_vs_readmission.png',  dpi =150, bbox_inches = 'tight')
print("Saved : visualisations/04_numeric_vs_readmission.png")

# Calculate and display statistical differences
print("\n Mean difference between Readmitted and not Readmitted Patients")
print("-"*80)
for col in numeric_features:
    if col in df.columns:
        mean_not_readmit = df[df['readmitted_binary'] == 0][col].mean()
        mean_readmit = df[df['readmitted_binary'] == 1][col].mean()
        diff = mean_readmit - mean_not_readmit
        pct_diff = (diff/ mean_not_readmit * 100) if mean_not_readmit != 0 else 0

        print(f"{col}:")
        print(f"Not readmitted: {mean_not_readmit:.2f}")
        print(f"Readmitted: {mean_readmit:.2f}")
        print(f"Difference: {diff:+.2f} ({pct_diff:+.1f}%)")
        print()

# 6.3.2 Categorical Features vs Readmission
print('\n\n 3.2 CATEGORICAL FEATURES VS READMISSION')
print("-"*80)

# Readmission rate by categorical features
categorical_features = ['gender', 'race','age','diag_1_category','diabetesMed','change']

fig,axes = plt.subplots(2,3,figsize=(18,10))
fig.suptitle("Readmission Rates by Categorical Features", fontsize =16, fontweight='bold')

for idx, col in enumerate(categorical_features):
    if col in df.columns:
        row = idx // 3
        col_idx = idx%3
        ax = axes [row, col_idx]

        # Calculate readmission rate by category
        readmit_rate = df.groupby(col)['readmitted_binary'].agg(['mean','count'])
        readmit_rate = readmit_rate[readmit_rate['count'] >= 100]
        readmit_rate = readmit_rate.sort_values('mean',ascending= False).head(10)

        # Create bar plot
        bars = ax.barh(range(len(readmit_rate)), readmit_rate['mean']* 100,
                  color = 'steelblue', alpha= 0.7)
        ax.set_yticks(range(len(readmit_rate)))
        ax.set_yticklabels(readmit_rate.index, fontsize =9)
        ax.set_xlabel('Readmission Rate(%)', fontsize = 10)
        ax.set_title(f'{col}',fontsize = 11, fontweight= 'bold')
        ax.grid(axis ='x', alpha=0.3)

        # Add percentage labels
        for i,(rate,count) in enumerate(zip(readmit_rate['mean'], readmit_rate['count'])):
            ax.text(rate*100,i,f'{rate*100:.1f}% (n={count:,})',
                   va ='center', fontsize =8)

plt.tight_layout()
plt.savefig('visualisations/05_categorical_vs_readmission.png')
plt.close('all')

# 6.3.3 Correlation Analysis
print('\n \n 3.3 CORRELATION ANALYSIS')
print('-'*80)

# Select numeric columns for correlation
corr_columns = [col for col in numeric_features if col in df.columns]
corr_columns.append('readmitted_binary')

# Calculate correlation Matrix
correlation_matrix = df[corr_columns].corr()

# Create Heatmap
plt.figure(figsize = (12,10))
sns.heatmap(correlation_matrix, annot = True, fmt ='.2f', cmap='coolwarm',
          center =0, square = True, linewidths =1 , cbar_kws={"shrink":0.8})
plt.title('Correlation Matrix - Numeric Features vs Readmission', fontsize = 14, fontweight='bold',pad=20)
plt.tight_layout()
plt.savefig('visualisations/06_correlation_matrix.png')
print("Saved: visualisations/06_correlation_matrix.png ")
plt.close('all')

# Print top correlations with target
print("\n Top Correlations with Readmission:")
print("-"*80)
target_corr = correlation_matrix['readmitted_binary'].drop('readmitted_binary').sort_values(
    key = abs, ascending = False
)
print(target_corr.to_string())

In [ ]:
# =========================================
# SECTION 7: KEY INSIGHTS FROM EDA
# =========================================

print("\n\n" + "="*80)
print("KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("="*80)

# 7.1 Calculate key insights for statistics
print("\n1.PATIENT DEMOGRAPHICS")
print("-"*80)

if 'gender' in df.columns:
    print(f"\n Age Statistics:")
    print(f" - Mean Age: {df['age_numeric'].mean():.1f} years")
    print(f" - Median Age: {df['age_numeric'].median():.1f}years")
    print(f" - Patients over 60: {(df['age_numeric'] >= 60).sum() / len(df)*100:.1f}%")

print("\n 2. READMISSION STATISTICS")
print("="*80)

total_patients = len(df)
readmitted_30 = (df['readmitted_binary']==1).sum()
readmission_rate = readmitted_30 / total_patients*100

print(f"Overall 30 day Readmission Rate: {readmission_rate:.2f} %")
print(f"Total Readmitted (<30 days): {readmitted_30:,} out of {total_patients:,} ")

In [ ]:
# =====================================================
# SECTION 8: PREDICTIVE MODELLING - DATA PREPARATION
# =====================================================

"""
Now we'll build machine learning models to predict 30-day readmission.
Steps:
1. Feature Selection and encoding
2. Train-Test split
3. Feature Scaling
4. Model training and evaluation
"""

print("\n\n" + "=" * 80)
print("PREDICTIVE MODELLING")
print("="*80)

print("\n 1. FEATURE SCALING AND ENGINEERING")
print("-"* 80)

# Select features for modelling
# We'll use a mix of demographic, clinical, and administrative features

# Define Feature Groups
demographic_features = ['age_numeric','gender','race']

clinical_features = [
    'time_in_hospital','num_lab_procedures','num_procedures','num_medications',
    'number_diagnoses','diag_1_category','diag_2_category','diag_3_category'
]

utilization_features = ['number_outpatient','number_emergency','number_inpatient']

medication_features = ['diabetesMed', 'change', 'insulin','metformin']

# Combines all features
model_features = (demographic_features + clinical_features + 
                 utilization_features + medication_features)

# Filter to only include features that exist in our dataset
model_features = [f for f in model_features if f in df.columns]

print(f"Total features selected: {len(model_features)}")
print(f"\n Feature Categories:")
print(f" - Demographic: {len([f for f in demographic_features if f in model_features])}")
print(f" - Clinical: {len([f for f in clinical_features if f in model_features])}")
print(f" - Utilization: {len([f for f in utilization_features if f in model_features])}")
print(f" - Medication: {len([f for f in medication_features if f in model_features])}")

# Create modelling Dataset
print("\n 2. CREATING MODELLING DATASET")
print("-"*80)

# Remove any rows with missing target
df_model = df[model_features + ['readmitted_binary']].copy()
df_model = df_model.dropna(subset=['readmitted_binary'])

print(f"Initial Dataset size: {len(df):,}")
print(f"Modelling Dataset size:{len(df_model):,}")
print(f"Rows dropped:{len(df) - len(df_model):,}")

# Seperate features and Target
x = df_model[model_features].copy()
y = df_model['readmitted_binary'].copy()

print(f"\n Feature Matrix (X) shape: {x.shape}")
print(f"Target Vector (Y) shape : {y.shape}")
print(f"Target Distribution: {y.value_counts().to_dict()}")

# 3. ENCODE CATEGORICAL VARIABLES
print("\n 3. ENCODING CATEGORICAL VARIABLES")
print("-"* 80)

# Identify categorical columns
categorical_cols = x.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns to encode: {len(categorical_cols)}")

# Use Label Encoding for tree-based models (simpler, works well for random forest)
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    x[col] = le.fit_transform(x[col].astype(str))
    label_encoders[col] = le
    print(f' - Encoded {col}: {len(le.classes_)} unique values')

print("All Categorical Variables Encoded")

# Check for any missing values
missing_counts = x.isnull().sum().sum()
if missing_counts > 0:
    print(f"\n Warning: {missing_counts} missing values detected. Filling with median...")
    x = x.fillna(x.median())
else: 
    print("\n No missing values in feature matrix")

# 4. TRAIN-TEST SPLIT
print("\n 4. TRAIN-TEST SPLIT")
print("="*80)

# Split data : 80% training, 20% testing
# Use stratification to maintain class balance

x_train, x_test, y_train, y_test = train_test_split(
    x,y, test_size =0.2, random_state = 42, stratify = y
)

print(f"Training set size: {len(x_train):,} ({len(x_train)/len(x)*100:.1f}%)")
print(f"Testing set size: {len(x_test):,} ({len(x_test)/len(x)*100:.1f}%)")

print(f"\nTraining set class distribution:")
print(y_train.value_counts())
print(f"  - Class 0 (Not Readmitted): {(y_train==0).sum()/len(y_train)*100:.2f}%")
print(f"  - Class 1 (Readmitted): {(y_train==1).sum()/len(y_train)*100:.2f}%")

print(f"\nTest set class distribution:")
print(y_test.value_counts())
print(f"  - Class 0 (Not Readmitted): {(y_test==0).sum()/len(y_test)*100:.2f}%")
print(f"  - Class 1 (Readmitted): {(y_test==1).sum()/len(y_test)*100:.2f}%")

# 5. Feature Scaling
print("\n 5. FEATURE SCALING")
print("-" * 80)
print("Scaling features using StandardScaler (mean=0, std=1)")

# Initialize Scaler
scaler = StandardScaler()

# Fit on training data and transform both train and test
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

# Convert back to DataFrame for easier handling
x_train_scaled = pd.DataFrame(x_train_scaled, columns = x_train.columns, index= x_train.index)
x_test_scaled = pd.DataFrame(x_test_scaled, columns = x_test.columns, index = x_test.index)

print("Features scaled Successfully")
print(f"\n Sample scaled Values (first feature):")
print(f" - Mean:{x_train_scaled.iloc[:,0].mean():.6f}")
print(f" - Std Dev: {x_train_scaled.iloc[:,0].std():.6f}")

print("Data Preparation completed! Ready for model Training")

In [ ]:
# =============================================================
# SECTION 9: MODEL TRAINING AND EVALUATION
# =============================================================
print("\n \n "+"="* 80)
print("MODEL TRAINING")
print("="* 80)

"""
We'll train two models:
1. Logistic Regression (baseline, interpretable)
2. Random Forest (more complex, often better performance)
"""

# Dictionary to store models and results
models = {}
results = {}

# 9.1 Logistic Regression
print("\n1. LOGISTIC REGRESSION (Baseline Model)")
print("="* 80)

print("Training Logistic Regression with Class Weights.....")
lr_model = LogisticRegression(
    random_state =42,
    max_iter= 1000,
    class_weight = 'balanced',
    solver = 'lbfgs'
)
# Train the Model
lr_model.fit(x_train_scaled, y_train)
print("Model Trained Succesfully")

# Make predictions
lr_pred_train = lr_model.predict(x_train_scaled)
lr_pred_test = lr_model.predict(x_test_scaled)
lr_pred_proba_test = lr_model.predict_proba(x_test_scaled)[:,1]

# Store the model
models['Logistic Regression'] = lr_model

# Calculate Metrics
lr_metrics = {
     'train_accuracy': accuracy_score(y_train, lr_pred_train),
    'test_accuracy': accuracy_score(y_test, lr_pred_test),
    'precision': precision_score(y_test, lr_pred_test),
     'recall': recall_score(y_test, lr_pred_test),
    'f1': f1_score(y_test, lr_pred_test),
    'roc_auc': roc_auc_score(y_test, lr_pred_proba_test)
}
results['Logistic Regression'] = lr_metrics

print("\nLogistic Regression Performance:")
print(f"  - Training Accuracy: {lr_metrics['train_accuracy']:.4f}")
print(f"  - Test Accuracy: {lr_metrics['test_accuracy']:.4f}")
print(f"  - Precision: {lr_metrics['precision']:.4f}")
print(f"  - Recall: {lr_metrics['recall']:.4f}")
print(f"  - F1 Score: {lr_metrics['f1']:.4f}")
print(f"  - ROC AUC: {lr_metrics['roc_auc']:.4f}")

# 9.2 RANDOM FOREST
print("\n\n 2. RANDOM FOREST CLASSIFIER")
print("="* 80)

print("Training Random Forest with 100 trees....")
rf_model = RandomForestClassifier (
    n_estimators = 100,
    max_depth = 15,
    min_samples_split = 50,
    min_samples_leaf = 20,
    random_state = 42,
    class_weight = 'balanced',
    n_jobs = -1
)

# Train the model
rf_model.fit(x_train,y_train)
print("Model Trained Succesfully")

# Make Predictions
rf_pred_train = rf_model.predict(x_train)
rf_pred_test = rf_model.predict(x_test)
rf_pred_proba_test = rf_model.predict_proba(x_test)[:,1]

# Store Model
models['Random Forest'] = rf_model

# Calculate Metrics
rf_metrics = {
    'train_accuracy': accuracy_score(y_train, rf_pred_train),
    'test_accuracy': accuracy_score(y_test, rf_pred_test),
    'precision': precision_score(y_test, rf_pred_test),
    'recall': recall_score(y_test, rf_pred_test),
    'f1': f1_score(y_test, rf_pred_test),
    'roc_auc': roc_auc_score(y_test, rf_pred_proba_test)
}
results['Random Forest'] = rf_metrics

print("\n Random Forest Performance:")
print(f"  - Training Accuracy: {rf_metrics['train_accuracy']:.4f}")
print(f"  - Test Accuracy: {rf_metrics['test_accuracy']:.4f}")
print(f"  - Precision: {rf_metrics['precision']:.4f}")
print(f"  - Recall: {rf_metrics['recall']:.4f}")
print(f"  - F1 Score: {rf_metrics['f1']:.4f}")
print(f"  - ROC AUC: {rf_metrics['roc_auc']:.4f}")

# 9.3 Model Comparisons
print("\n \n 3. MODEL COMPARISONS")
print("="*80)

comparison_df = pd.DataFrame(results).T
print("\n Detailed Metrics Comparison:")
print(comparison_df.round(4))

# Determine best model
best_model_name = comparison_df['roc_auc'].idxmax()
print(f"\n Best Model (by ROC AUC): {best_model_name} ")
print(f" ROC AUC Score: {comparison_df.loc[best_model_name, 'roc_auc']:.4f}")

# Visualize Model Comparisons
fig, axes = plt.subplots(1,2, figsize =(14,5))

# Metric Comparison
metrics_to_plot = ['test_accuracy', 'precision', 'recall', 'f1', 'roc_auc']
comparison_df[metrics_to_plot].T.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Model Performance Comparison', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Score')
axes[0].set_xlabel('Metric')
axes[0].set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC'], rotation=45)
axes[0].legend(title='Model')
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0, 1])

# ROC Curves 
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_pred_proba_test)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_pred_proba_test)

axes[1].plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={lr_metrics["roc_auc"]:.3f})',
            linewidth=2, color='steelblue')
axes[1].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_metrics["roc_auc"]:.3f})',
            linewidth=2, color='coral')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves', fontweight='bold', fontsize=12)
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('visualisations/07_model_comparison.png', dpi =300, bbox_inches= 'tight')
print("Saved: visualisations/07_model_comparison.png ")
plt.close('all')

In [ ]:
# 9.4 DETAILED EVALUATION - CONFUSION MATRICES
print("\n \n 4. DETAILED MODEL EVALUATION")
print("="* 80)

# Select best model for detailed analysis
best_model = models[best_model_name]
if best_model_name == 'Logistic Regression':
    best_pred = lr_pred_test
    best_proba = lr_pred_proba_test

else:
    best_pred = rf_pred_test
    best_proba = rf_pred_proba_test

# Confusion Matrices
fig, axes = plt.subplots(1,2, figsize=(14,5))

# Logistic Regression confusion matrix
cm_lr = confusion_matrix(y_test,lr_pred_test)
sns.heatmap (cm_lr, annot = True, fmt='d', cmap= 'Blues', ax= axes[0],
            xticklabels = ['Not Readmitted', 'Readmitted'],
            yticklabels = ['Not Readmitted', 'Readmitted'])
axes[1].set_title('Random forest \n Confusion Matrix', fontweight ='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel ("Predicted Label")

plt.tight_layout()
plt.savefig('visualisations/08_confusion_matrices.png', dpi= 300,bbox_inches = 'tight')
print("Saved : visualisations/08_confusion_matrices.png")
plt.close('all')

# Print Classification Reports
print(f"\n {best_model_name} - Detailed Classification Reports:")
print("="* 80)
print(classification_report(y_test, best_pred, target_names=['Not Readmitted', 'Readmitted']))

# 9.5 FEATURE IMPORTANCE ANALYSIS
print("\n \n 5. FEATURE IMPORTANCE ANALYSIS")
print("="* 80)

if best_model_name == "Random Forest":
    # Get feature importances from Random Forest
    feature_importance = pd.DataFrame({
        'feature':x_train.columns,
        'importance' : rf_model.feature_importances_
    }).sort_values('importance', ascending = False)

    print("\n Top 15 most Important Features:")
    print(feature_importance.head(15).to_string(index=False))

    # Visualize feature importance
    plt.figure(figsize=(12,8))
    top_features = feature_importance.head(20)
    plt.barh(range(len(top_features)), top_features['importance'], color = 'coral')
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel ('Feature Importance', fontweight='bold')
    plt.title('Top 20 Feature Importances - Random Forest',
             fontweight ='bold', fontsize = 14 )
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('visualisations/09_feature_importance.png', dpi=300, bbox_inches='tight')
    print("\n✓ Saved: visualisations/09_feature_importance.png")

else:
     # Get coefficients from Logistic Regression
    feature_coef = pd.DataFrame({
        'feature': X_train_scaled.columns,
        'coefficient': lr_model.coef_[0]
    }).sort_values('coefficient', key=abs, ascending=False)
    
    print("\nTop 15 Features by Coefficient Magnitude:")
    print(feature_coef.head(15).to_string(index=False))
    
    # Visualize coefficients
    plt.figure(figsize=(12, 8))
    top_features = feature_coef.head(20)
    colors = ['red' if x < 0 else 'green' for x in top_features['coefficient']]
    plt.barh(range(len(top_features)), top_features['coefficient'], color=colors, alpha=0.7)
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Coefficient Value', fontweight='bold')
    plt.title('Top 20 Features by Coefficient - Logistic Regression', 
              fontweight='bold', fontsize=14)
    plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('visualisations/09_feature_importance.png', dpi=300, bbox_inches='tight')
    print("\n✓ Saved: visualisations/09_feature_importance.png") 

print("Model Training and Evaluation Completed")

# 9.6 PREDICTION PROBABILITY DISTRIBUTION
print("\n \n 6. PREDICTION PROBABILITY DISTRIBUTION")
print("-"* 80)

# Analyze how confident the model is on its Predictions
plt.figure(figsize = (12,6))

# Plot for Actual Negatives
plt.subplot(1, 2, 1)
not_readmit_probs = best_proba[y_test == 0]
plt.hist(not_readmit_probs, bins=50, alpha=0.7, color='green', edgecolor='black')
plt.xlabel('Predicted Probability of Readmission')
plt.ylabel('Count')
plt.title('Prediction Distribution\nActual: Not Readmitted', fontweight='bold')
plt.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold')
plt.legend()
plt.grid(axis='y', alpha=0.3)

# Plot for actual positives
plt.subplot(1, 2, 2)
readmit_probs = best_proba[y_test == 1]
plt.hist(readmit_probs, bins=50, alpha=0.7, color='coral', edgecolor='black')
plt.xlabel('Predicted Probability of Readmission')
plt.ylabel('Count')
plt.title('Prediction Distribution\nActual: Readmitted', fontweight='bold')
plt.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold')
plt.legend()
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('visualisations/10_probability_distribution.png', dpi =300, bbox_inches = 'tight')
print("Saved : visualisations/10_probability_distribution.png")
plt.close('all')

# 9.7 BUISNESS METRICS AND COST-BENEFIT ANALYSIS
print("\n \n 7. BUISNESS IMPACT ANALYSIS")
print("-"* 80)

"""
In healthcare, different types of errors have different costs:
- False Negative (FN): Missing a readmission - patient returns, no intervention
- False Positive (FP): False alarm - unnecessary intervention/resources
- True Positive (TP): Correctly identifying at-risk patient - successful intervention
- True Negative (TN): Correctly identifying low-risk patient - no intervention needed
"""

cm = confusion_matrix(y_test,best_pred)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix Breakdown:")
print(f" - True Negatives (TN):{tn:,} - Correctly Predicted No Readmission")
print(f" - False Positives (FP):{fp:,} - Incorrectly Predicted Readmission")
print(f" - False Negatives (FN):{fn:,} - Missed Readmissions (High Cost)")
print(f" - True POsitives (TP):{tp:,} - Correctly Predicted Readmissions")

# Calculate Business Metrics
total_readmissions = tp + fn
total_predictions = tn + fp + fn + tp

print(f"\n Buisness Metrics:")
print(f" - Total Actual Readmissions :{total_readmissions:,}")
print(f" - Readmissions caught by Model: {tp:,} ({tp/total_readmissions*100:.1f}%)")
print(f" - Readmissions missed by Model: {fn:,} ({fn/total_readmissions*100:.1f}%)")

# Cost-benefit analysis
cost_readmission = 15000
cost_intervention = 1000
savings_per_prevented = cost_readmission * 0.7

# Calculate Potential Savings
prevented_readmissions = tp * 0.7 
intervention_cost = (tp+fp) * cost_intervention
readmissions_cost_saved = prevented_readmissions * cost_readmission
net_benefit = readmissions_cost_saved - intervention_cost

print(f"\nCost-Benefit Analysis (Hypothetical):")
print(f"  - Patients flagged for intervention: {tp + fp:,}")
print(f"  - Cost of interventions: ${intervention_cost:,.0f}")
print(f"  - Estimated readmissions prevented: {prevented_readmissions:.0f}")
print(f"  - Estimated cost savings: ${readmissions_cost_saved:,.0f}")
print(f"  - Net benefit: ${net_benefit:,.0f}")

if net_benefit > 0:
    print(f"\n Model Provides positive ROI: ${net_benefit:,.0f} net savings")
else:
    print(f"\n Model needs improvement for positive ROI")

# 9.8 - RISK STRATIFIACTION
print("\n \n 8. PATIENT RISK STRATIFICATION")
print("-" * 80)

# Classify patients into risks categories based on prediction probability
risk_categories = pd.cut(best_proba,
                        bins = [0,0.3,0.5,0.7,1.0],
                        labels = ['Low Risk', 'Medium Risk', 'High Risk','Very High Risk'])

risk_summary = pd.DataFrame({
    'Risk Category': risk_categories,
    "Actual Readmission": y_test.values
})

risk_stats = risk_summary.groupby('Risk Category').agg({
    'Actual Readmission' : ['count','sum','mean']
})
risk_stats.columns = ['Total Patients','Actual Readmissions','Readmission Rate']

print("\n Patient Risk Stratification: ")
print(risk_stats.to_string())

# Visualise risk Stratification
plt.figure(figsize=(12,6))

plt.subplot(1, 2, 1)
risk_stats['Total Patients'].plot(kind='bar', color='steelblue', alpha=0.7)
plt.title('Patient Distribution by Risk Category', fontweight='bold')
plt.xlabel('Risk Category')
plt.ylabel('Number of Patients')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
risk_stats['Readmission Rate'].plot(kind='bar', color='coral', alpha=0.7)
plt.title('Actual Readmission Rate by Risk Category', fontweight='bold')
plt.xlabel('Risk Category')
plt.ylabel('Readmission Rate')
plt.xticks(rotation=45)
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)

# Add Percentage Labels
for i, v in enumerate(risk_stats['Readmission Rate']):
    plt.text(i,v + 0.02, f'{v*100:.1f}%', ha='center',fontweight = 'bold')

plt.tight_layout()
plt.savefig('visualisations/11_risk_stratification.png', dpi= 300, bbox_inches = 'tight')
print("\n Saved: visualisations/11_risk_stratification.png")
plt.close('all')

print("\n Predictive Modelling Completed!")

In [ ]:
# ==============================================================
# SECTION 10 - EXECUTIVE SUMMARY AND REPORTING
# ==============================================================

print("\n\n" + "=" *80)
print("EXECUTIVE SUMMARY GENERATION")
print("="* 80)

# Generate comprehensive summary report
summary_report = f"""
{'='* 80}
HEALTHCARE READMISSION PREDICTION - EXECUTIVE SUMMARY 
{'=' * 80}
Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

PROJECT OVERVIEW
{'-' * 80}
Objective: Predict 30-day hospital readmissions data for diabetes patients
Dataset: Diabetes 130-US Hospitals (1999-2000)
Total Records: {len(df):,}
Analysis Period: Complete end-to-end analytics Pipeline

KEY FINDINGS 
{'-' * 80}
1. READMISSION STATISTICS
- Overall 30-day readmission rate: {readmission_rate:.2f}%
- Total Readmissions in dataset: {readmitted_30:,}
- Patient Population: Primarily Elderly (median age {df['age_numeric'].median():.0f} years)

2. RISK FACTORS IDENTIFIED
- Prior inpatient admissions: Strong predictor of readmission
- Emergency department utilization: Higher visits correlate with readmission
- Diagnosis category: Circulatory and respiratory conditions show elevated risk
- Age: Elderly patients (70+) have higher readmission rates

3. MODEL PERFORMANCE
- Best Model: {best_model_name}
- Test Accuracy: {results[best_model_name]['test_accuracy']:.2%}
- Precision: {results[best_model_name]['precision']:.2%}
- Recall: {results[best_model_name]['recall']:.2%}
- F1 Score: {results[best_model_name]['f1']:.2%}
- ROC AUC: {results[best_model_name]['roc_auc']:.4f} 

CONFUSION MATRIX ANALYSIS
{'-'* 80}
    True Negatives : {tn:,} - Correct low-risk predictions
    False Positives : {fp:,} - Unnecessary interventions
    False Negatives : {fn:,} - Missed Readmissions (critical)
    True Positives : {tp:,} - Succesful High-Risk Identification

    Model Captures {tp/total_readmissions*100:.1f}% of actual readmissions

BUISNESS IMPACT (HYPOTHETICAL SCENARIO)
{'-' * 80}
     Assumptions:
     - Average Readmission Cost : ${cost_readmission:,}
     - Intervention cost per patient : ${cost_intervention:,}
     - Intervention Effectiveness : 70%

     Projected Outcomes:
   - Patients requiring intervention: {tp + fp:,}
   - Estimated readmissions prevented: {prevented_readmissions:.0f}
   - Total intervention cost: ${intervention_cost:,.0f}
   - Potential cost savings: ${readmissions_cost_saved:,.0f}
   - Net benefit: ${net_benefit:,.0f}

PATIENT RISK STRATIFICTAION
{'-'* 80}
"""

# Add risk stratification details
for risk_level in risk_stats.index:
    count = risk_stats.loc[risk_level, 'Total Patients']
    rate = risk_stats.loc[risk_level, 'Readmission Rate']
    summary_report += f" {risk_level}: {count:.0f} patients({ rate*100:.1f}% readmission rate)\n"

    summary_report += f"""

TECHNICAL SPECIFICATIONS
{'-'*80}
- Programming Language: Python 3.x
- Key Libraries: pandas, scikit-learn, matplotlib, seaborn
- Model Type: {best_model_name}
- Features Used: {len(model_features)} clinical and demographic variables
- Training Data: {len(x_train):,} records
- Test Data: {len(x_test):,} records
- Cross-validation: Stratified train-test split (80/20)

{"=" *80}
END OF EXECUTIVE SUMMARY
{'=' * 80}
"""

# Print the Summary
print(summary_report)

# Save summary to file
with open ('EXECUTIVE_SUMMARY.txt', 'w') as f:
    f.write(summary_report)

print("\n Executive summary saved to: EXECUTIVE_SUMMARY.txt")

In [ ]:
# =================================================
# SECTION 11 : FINAL DASHBOARD VISUALIZATION
# =================================================

print("\n \n" + "="*50)
print("CREATING FINAL DASHBOARD")
print("=" * 50)

# Create a comprehensive dashboard with all key metrics
fig = plt.figure(figsize=(20,12))
gs = fig.add_gridspec(3,3, hspace =0.3, wspace = 0.3)

# 1. Readmission Rate Overview
ax1 = fig.add_subplot(gs[0, 0])
readmit_counts = df['readmitted'].value_counts()
colors_pie = ['#2ecc71', '#e74c3c', '#3498db']
ax1.pie(readmit_counts.values, labels=readmit_counts.index, autopct='%1.1f%%',
        colors=colors_pie, startangle=90)
ax1.set_title('Overall Readmission Status', fontweight='bold', fontsize=12)

# 2. Age Distribution
ax2 = fig.add_subplot(gs[0, 1])
age_order = sorted(df['age'].unique())
age_data = df['age'].value_counts().reindex(age_order)
ax2.bar(range(len(age_data)), age_data.values, color='mediumpurple', alpha=0.7)
ax2.set_xticks(range(len(age_data)))
ax2.set_xticklabels(age_data.index, rotation=45, ha='right', fontsize=8)
ax2.set_title('Patient Age Distribution', fontweight='bold', fontsize=12)
ax2.set_ylabel('Count')
ax2.grid(axis='y', alpha=0.3)

# 3. Top Diagnoses
ax3 = fig.add_subplot(gs[0, 2])
if 'diag_1_category' in df.columns:
    top_diag = df['diag_1_category'].value_counts().head(8)
    ax3.barh(range(len(top_diag)), top_diag.values, color='coral', alpha=0.7)
    ax3.set_yticks(range(len(top_diag)))
    ax3.set_yticklabels(top_diag.index, fontsize=9)
    ax3.set_title('Top 8 Primary Diagnoses', fontweight='bold', fontsize=12)
    ax3.set_xlabel('Count')
    ax3.grid(axis='x', alpha=0.3)

# 4. Model Performance Comparison
ax4 = fig.add_subplot(gs[1, 0])
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC']
lr_scores = [lr_metrics['test_accuracy'], lr_metrics['precision'], 
             lr_metrics['recall'], lr_metrics['f1'], lr_metrics['roc_auc']]
rf_scores = [rf_metrics['test_accuracy'], rf_metrics['precision'],
             rf_metrics['recall'], rf_metrics['f1'], rf_metrics['roc_auc']]

x = np.arange(len(metrics))
width = 0.35
ax4.bar(x - width/2, lr_scores, width, label='Logistic Reg', color='steelblue', alpha=0.8)
ax4.bar(x + width/2, rf_scores, width, label='Random Forest', color='coral', alpha=0.8)
ax4.set_ylabel('Score')
ax4.set_title('Model Performance Metrics', fontweight='bold', fontsize=12)
ax4.set_xticks(x)
ax4.set_xticklabels(metrics, rotation=45, ha='right')
ax4.legend()
ax4.set_ylim([0, 1])
ax4.grid(axis='y', alpha=0.3)

# 5. Confusion Matrix (Best Model)
ax5 = fig.add_subplot(gs[1, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=ax5,
            xticklabels=['Not Readmit', 'Readmit'],
            yticklabels=['Not Readmit', 'Readmit'],
            cbar_kws={'label': 'Count'})
ax5.set_title(f'{best_model_name}\nConfusion Matrix', fontweight='bold', fontsize=12)
ax5.set_ylabel('True Label')
ax5.set_xlabel('Predicted Label')

# 6. ROC Curve
ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(fpr_lr, tpr_lr, label=f'LR (AUC={lr_metrics["roc_auc"]:.3f})',
         linewidth=2.5, color='steelblue')
ax6.plot(fpr_rf, tpr_rf, label=f'RF (AUC={rf_metrics["roc_auc"]:.3f})',
         linewidth=2.5, color='coral')
ax6.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random')
ax6.set_xlabel('False Positive Rate', fontweight='bold')
ax6.set_ylabel('True Positive Rate', fontweight='bold')
ax6.set_title('ROC Curves', fontweight='bold', fontsize=12)
ax6.legend(loc='lower right')
ax6.grid(alpha=0.3)

# 7. Feature Importance
ax7 = fig.add_subplot(gs[2, :2])
if best_model_name == 'Random Forest':
    top_15 = feature_importance.head(15)
    ax7.barh(range(len(top_15)), top_15['importance'], color='teal', alpha=0.7)
    ax7.set_yticks(range(len(top_15)))
    ax7.set_yticklabels(top_15['feature'], fontsize=9)
    ax7.set_xlabel('Importance Score', fontweight='bold')
    ax7.set_title('Top 15 Feature Importances', fontweight='bold', fontsize=12)
    ax7.invert_yaxis()
    ax7.grid(axis='x', alpha=0.3)

# 8. Risk Stratification
ax8 = fig.add_subplot(gs[2, 2])
risk_rates = risk_stats['Readmission Rate'].values
risk_labels = risk_stats.index.tolist()
colors_risk = ['green', 'yellow', 'orange', 'red']
bars = ax8.bar(range(len(risk_rates)), risk_rates, color=colors_risk, alpha=0.7,
               edgecolor='black', linewidth=1.5)
ax8.set_xticks(range(len(risk_rates)))
ax8.set_xticklabels(risk_labels, rotation=45, ha='right', fontsize=9)
ax8.set_ylabel('Readmission Rate', fontweight='bold')
ax8.set_title('Risk Stratification Performance', fontweight='bold', fontsize=12)
ax8.set_ylim([0, 1])
ax8.grid(axis='y', alpha=0.3)

# Add percentage Labels
for i, v in enumerate(risk_rates):
    ax8.text(i,v + 0.02, f'{v*100:.1f}%', ha = 'center',fontweight = 'bold')

# Overall Title
fig.suptitle("Healthcare Readmission Prediction - Analytics Dashboard",
            fontsize = 18, fontweight= 'bold', y=0.995)

plt.savefig('visualisations/12_FINAL_DASHBOARD.png', dpi =300, bbox_inches ='tight')
print('Saved : visualisations/12_FINAL_DASHBOARD.png')
# plt.show()
plt.close('all')

print("\n Final Dashboard created succesfully")

In [ ]:
# Create a Project Completion Report
print("\n \n" + "="*80)
print("PROJECT COMPLETION REPORT ")
print("=" * 80)

# Calculate total execution metrics
end_time = datetime.now()
print(f"\n Project Execution Completed at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Summary Statistics
print("\n" + "=" * 80)
print("DATA PROCESSING SUMMARY")
print("=" * 80)
print(f"Original dataset size: {len(df_raw):,} rows × {df_raw.shape[1]} columns")
print(f"Cleaned dataset size: {len(df):,} rows × {df.shape[1]} columns")
print(f"Features engineered: {df.shape[1] - df_raw.shape[1]} new features created")
print(f"Missing values handled: Yes")
print(f"Categorical encoding: {len(categorical_cols)} features encoded")

print("\n" + "-" * 80)
print("ANALYSIS SUMMARY")
print("-" * 80)
print(f"Target variable: 30-day readmission (binary)")
print(f"Readmission rate: {readmission_rate:.2f}%")
print(f"Total patients analyzed: {len(df):,}")
print(f"Readmitted patients: {readmitted_30:,}")
print(f"Not readmitted patients: {len(df) - readmitted_30:,}")

print("\n" + "-" * 80)
print("MODEL TRAINING SUMMARY")
print("-" * 80)
print(f"Models trained: {len(models)}")
print(f"  1. Logistic Regression")
print(f"  2. Random Forest Classifier")
print(f"\nBest performing model: {best_model_name}")
print(f"  - ROC AUC: {results[best_model_name]['roc_auc']:.4f}")
print(f"  - Accuracy: {results[best_model_name]['test_accuracy']:.4f}")
print(f"  - Precision: {results[best_model_name]['precision']:.4f}")
print(f"  - Recall: {results[best_model_name]['recall']:.4f}")

print("\n" + "-" * 80)
print("OUTPUTS GENERATED")
print("-" * 80)
print(" Cleaned dataset: diabetic_data_cleaned.csv")
print(" Executive summary: EXECUTIVE_SUMMARY.txt")
print("Project documentation: README.md")
print(" Dependencies list: requirements.txt")
print(" Git configuration: .gitignore")
print("\n Visualisations (12 files):")
visualisation_files = [
    "01_target_distribution.png",
    "02_numeric_distributions.png",
    "03_categorical_distributions.png",
    "04_numeric_vs_readmission.png",
    "05_categorical_vs_readmission.png",
    "06_correlation_matrix.png",
    "07_model_comparison.png",
    "08_confusion_matrices.png",
    "09_feature_importance.png",
    "10_probability_distribution.png",
    "11_risk_stratification.png",
    "12_FINAL_DASHBOARD.png"
]
for vis_file in visualisation_files:
    print(f" - visualisations/{vis_file}")

print("\n\n" + "=" * 80)
print("PROJECT SUCCESSFULLY COMPLETED!")
print("=" * 80)

print("=" * 80)
print(f"End Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

In [ ]:
cd HEALTHCARE READMISSION DATA.ipynb

In [ ]:
git init


In [ ]:
os.getcwd()


In [ ]:
!git init

In [ ]:
!git add .

In [ ]:
!git commit -m "Initial commit"

In [ ]:
git remote add origin https://github.com/Khushinkm15/healthcare_readmission_prediction.git

In [ ]:
git branch -M main

In [ ]:
git push -u origin main

In [ ]:
os.getcwd()

In [ ]:
os.listdir()

In [ ]:
!git log

In [ ]:
!git status